# Tema 3 – Rețele Bayesiene

## Exercițiul 1 – Problema Monty Hall

### 1. Descrierea problemei

Problema Monty Hall este o problemă de probabilitate bazată pe un joc cu trei uși. În spatele unei singure uși se află un premiu (o mașină), iar în spatele celorlalte două se află câte o capră.

Concurentul alege inițial o ușă. Prezentatorul, care știe unde se află premiul, deschide una dintre celelalte uși, dezvăluind o capră. Concurentul poate păstra ușa aleasă inițial sau poate schimba alegerea cu cealaltă ușă rămasă închisă.

În exemplul din documentația `pgmpy`, ușile sunt numerotate de la **0 la 2**. Sunt folosite trei variabile aleatoare:

- `C` – ușa aleasă de concurent;
- `P` – ușa în spatele căreia se află premiul;
- `H` – ușa deschisă de prezentator.

### 2. Structura rețelei Bayesiene

Rețeaua are muchiile `C → H` și `P → H`. Alegerea prezentatorului (`H`) depinde de ușa aleasă de concurent (`C`) și de poziția premiului (`P`).

### 3. Importarea bibliotecilor

Importăm `DiscreteBayesianNetwork` pentru a defini structura rețelei, `TabularCPD` pentru distribuțiile de probabilitate condiționată și `VariableElimination` pentru inferență.


In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination


### 4. Definirea structurii rețelei

Creăm rețeaua cu două muchii: `C → H` și `P → H`. Astfel, `C` și `P` sunt părinții variabilei `H`.


In [ ]:
model = DiscreteBayesianNetwork([("C", "H"), ("P", "H")])


### 5. Definirea distribuțiilor de probabilitate (CPD)

Variabilele `C` și `P` au câte trei valori posibile (0, 1, 2). Alegerea concurentului și poziția premiului sunt uniforme, deci folosim probabilitatea exactă `1/3` pentru fiecare ușă.

Pentru `H`, tabelul specifică probabilitatea ca prezentatorul să deschidă fiecare ușă, în funcție de `C` și `P`. Valorile respectă regula că prezentatorul nu deschide ușa aleasă de concurent și nici ușa care ascunde premiul. În unele situații poate alege între două uși cu capre, fiecare cu probabilitatea `0.5`.


In [ ]:
cpd_c = TabularCPD("C", 3, [[1/3], [1/3], [1/3]])
cpd_p = TabularCPD("P", 3, [[1/3], [1/3], [1/3]])

cpd_h = TabularCPD(
    variable="H",
    variable_card=3,
    values=[
        [0, 0, 0, 0, 0.5, 1, 0, 1, 0.5],
        [0.5, 0, 1, 0, 0, 0, 1, 0, 0.5],
        [0.5, 1, 0, 1, 0.5, 0, 0, 0, 0],
    ],
    evidence=["C", "P"],
    evidence_card=[3, 3],
)


### 6. Asocierea CPD-urilor și verificarea modelului

Adăugăm distribuțiile de probabilitate în rețea. Metoda `check_model()` verifică dacă structura și distribuțiile asociate sunt consistente. Rezultatul așteptat este `True`.


In [ ]:
model.add_cpds(cpd_c, cpd_p, cpd_h)

print("Distribuțiile de probabilitate definite:")
print(model.get_cpds())

print("Model valid:", model.check_model())


### 7. Inferența probabilistică

Presupunem că participantul a ales ușa `0`, iar prezentatorul a deschis ușa `2`. Calculăm distribuția posterioară a poziției premiului, adică `P(P | C=0, H=2)`, folosind algoritmul `VariableElimination`.


In [ ]:
infer = VariableElimination(model)

posterior_p = infer.query(
    variables=["P"],
    evidence={"C": 0, "H": 2},
)

print(posterior_p)


### 8. Interpretarea rezultatului

Rezultatul așteptat este aproximativ:

- `P = 0`: `0.3333`;
- `P = 1`: `0.6667`;
- `P = 2`: `0.0000`.

Premiul nu poate fi în spatele ușii `2`, deoarece prezentatorul a deschis acea ușă și a dezvăluit o capră. Ușa `1` are probabilitatea posterioară cea mai mare de a ascunde premiul.

### 9. Concluzie

Problema Monty Hall poate fi reprezentată printr-o rețea Bayesiană cu trei variabile: alegerea concurentului (`C`), poziția premiului (`P`) și ușa deschisă de prezentator (`H`). Variabilele `C` și `P` sunt părinții variabilei `H`, deoarece prezentatorul își alege ușa în funcție de alegerea concurentului și de poziția premiului.

Folosind distribuțiile de probabilitate condiționată și algoritmul de inferență `VariableElimination`, calculăm probabilitatea poziției premiului după observarea ușii alese și a ușii deschise. În exemplul analizat, probabilitatea ca premiul să fie în spatele ușii `1` este aproximativ `2/3`, iar probabilitatea ca acesta să fie în spatele ușii `0` este aproximativ `1/3`. Prin urmare, schimbarea alegerii inițiale oferă o probabilitate mai mare de câștig.

**Notă:** documentația afișează `0.33` pentru probabilitățile uniforme; în acest notebook folosim `1/3` pentru a păstra distribuțiile normalizate exact.
